In [ ]:
#pip install psycopg2-binary pandas
#pip install python-dotenv


SyntaxError: invalid syntax (3350084567.py, line 2)

In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import os

# 0. Charger les variables du fichier .env

load_dotenv()

PGHOST = os.getenv("PGHOST")
PGDATABASE = os.getenv("PGDATABASE")
PGUSER = os.getenv("PGUSER")
PGPASSWORD = os.getenv("PGPASSWORD")
PGPORT = os.getenv("PGPORT")



# 1. Charger le CSV
df = pd.read_csv("../data/processed/final_clean.csv")

# 2. Harmoniser les noms de colonnes entre CSV et table SQL
df = df.rename(columns={
    "unemployment_rate_": "unemployment_rate"
})

# 3. Remplacer les valeurs vides par None (NULL SQL)
df = df.where(pd.notnull(df), None)

# 4. Connexion à Neon
conn = psycopg2.connect(
    dbname=PGDATABASE,
    user=PGUSER,
    password=PGPASSWORD,
    host=PGHOST,
    port=PGPORT,
    sslmode="require"
)

cursor = conn.cursor()

# 5. Colonnes dans l'ordre exact de ta table
columns = [
    "year",
    "country_name",
    "life_evaluation_3_year_average",
    "lower_whisker",
    "upper_whisker",
    "explained_by_log_gdp_per_capita",
    "explained_by_social_support",
    "explained_by_healthy_life_expectancy",
    "explained_by_freedom_to_make_life_choices",
    "explained_by_generosity",
    "explained_by_perceptions_of_corruption",
    "dystopia_residual",
    "key_iso3_year",
    "inflation_cpi",
    "gdp_current_usd",
    "gdp_per_capita_current_usd",
    "unemployment_rate",
    "interest_rate_real",
    "inflation_gdp_deflator",
    "gdp_growth_annual",
    "current_account_balance_gdp",
    "government_expense_of_gdp",
    "government_revenue_of_gdp",
    "tax_revenue_of_gdp",
    "gross_national_income_usd",
    "public_debt_of_gdp",
    "country_code"
]

# 6. Extraire les valeurs dans le bon ordre
values = [tuple(row[col] for col in columns) for _, row in df.iterrows()]

# 7. Requête d'insertion
insert_query = f"""
INSERT INTO world_happiness ({", ".join(columns)})
VALUES %s
"""

# 8. Insertion en batch (rapide)
execute_values(cursor, insert_query, values)

conn.commit()
cursor.close()
conn.close()

print("Import terminé avec succès.")


Import terminé avec succès.
